# 🧠 EEG-Based Emotion Recognition on DEAP Dataset
### A World-Class ML Pipeline: Signal Processing → Deep Learning → Explainability

---
> **Dataset:** DEAP (Database for Emotion Analysis using Physiological Signals)  
> **Signals:** 32-channel EEG + 8 peripheral signals @ 128 Hz  
> **Subjects:** 32 participants (s01.dat → s32.dat)  
> **Trials:** 40 music video clips per subject  
> **Labels:** Valence, Arousal, Dominance, Liking (1–9 continuous scale)

---

## 🗺️ Pipeline Overview
```
Raw .dat Files
     │
     ▼
1. Data Loading & Structure Exploration
     │
     ▼
2. Exploratory Data Analysis (EDA)
   ├── Label distributions (Valence / Arousal)
   ├── Raw EEG time-series visualization
   ├── Per-channel signal statistics
   └── Correlation heatmaps
     │
     ▼
3. Signal Preprocessing
   ├── Bandpass filter (4–45 Hz)
   ├── Notch filter (50 Hz power-line)
   └── Baseline removal (pre-trial segment)
     │
     ▼
4. Feature Extraction
   ├── Band Power (Delta/Theta/Alpha/Beta/Gamma)
   ├── Fast Fourier Transform (FFT)
   ├── Differential Asymmetry (DASM) & Rational Asymmetry (RASM)
   ├── Statistical features (mean, std, skew, kurtosis)
   └── Hjorth Parameters (activity, mobility, complexity)
     │
     ▼
5. Dimensionality Reduction & Analysis
   ├── PCA (variance explained, 2D/3D scatter)
   ├── t-SNE visualization
   └── Feature importance (Random Forest)
     │
     ▼
6. Model Training & Comparison
   ├── SVM (RBF kernel)
   ├── Random Forest
   ├── XGBoost
   ├── 1D-CNN
   ├── LSTM / Bi-LSTM
   └── ★ CNN-LSTM Hybrid (Best Model)
     │
     ▼
7. Evaluation
   ├── Confusion Matrix
   ├── ROC-AUC Curves
   ├── Per-class Metrics
   └── Model Comparison Table
```


## 📦 Cell 1 — Seeds & Reproducibility

In [ ]:
import os, random
import numpy as np
import tensorflow as tf

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'

print(f"✅ All seeds locked to {SEED} — 100% reproducible results guaranteed")

## 📚 Cell 2 — All Imports

In [ ]:
# ── Core ──────────────────────────────────────────────────────────────────────
import warnings, pickle, glob
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm.notebook import tqdm
from collections import defaultdict

# ── Signal Processing ─────────────────────────────────────────────────────────
from scipy import signal
from scipy.stats import skew, kurtosis
from scipy.signal import butter, filtfilt, iirnotch, welch

# ── ML / Sklearn ──────────────────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, auc,
    accuracy_score, f1_score
)
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.pipeline import Pipeline

# ── XGBoost ───────────────────────────────────────────────────────────────────
try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print("⚠️  XGBoost not installed. Run: pip install xgboost")

# ── Deep Learning ─────────────────────────────────────────────────────────────
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Dense, Dropout, BatchNormalization, Flatten,
    Conv1D, MaxPooling1D, GlobalAveragePooling1D,
    LSTM, Bidirectional, TimeDistributed, Reshape,
    Input, Concatenate, Attention, MultiHeadAttention,
    LayerNormalization, Add
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.regularizers import l2

print("✅ All libraries loaded successfully")
print(f"   TensorFlow version : {tf.__version__}")
print(f"   NumPy version      : {np.__version__}")

## 🎨 Cell 3 — Global Plot Style

In [ ]:
# ── Publication-quality dark theme ────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor' : '#0d1117',
    'axes.facecolor'   : '#161b22',
    'axes.edgecolor'   : '#30363d',
    'axes.labelcolor'  : '#e6edf3',
    'text.color'       : '#e6edf3',
    'xtick.color'      : '#8b949e',
    'ytick.color'      : '#8b949e',
    'grid.color'       : '#21262d',
    'grid.linestyle'   : '--',
    'grid.alpha'       : 0.5,
    'axes.grid'        : True,
    'font.family'      : 'DejaVu Sans',
    'axes.titlesize'   : 14,
    'axes.labelsize'   : 12,
    'legend.facecolor' : '#161b22',
    'legend.edgecolor' : '#30363d',
})

PALETTE   = ['#58a6ff','#3fb950','#f78166','#d2a8ff','#ffa657','#79c0ff','#56d364']
CMAP_DIV  = 'RdYlGn'
CMAP_SEQ  = 'Blues'

# EEG channel names (DEAP uses 32-channel standard 10-20 system)
CHANNEL_NAMES = [
    'Fp1','AF3','F3','F7','FC5','FC1','C3','T7',
    'CP5','CP1','P3','P7','PO3','O1','Oz','Pz',
    'Fp2','AF4','Fz','F4','F8','FC6','FC2','Cz',
    'C4','T8','CP6','CP2','P4','P8','PO4','O2'
]

# Frequency bands (Hz)
BANDS = {
    'Delta' : (0.5, 4),
    'Theta' : (4,   8),
    'Alpha' : (8,  14),
    'Beta'  : (14, 30),
    'Gamma' : (30, 45)
}

SRATE = 128  # DEAP sampling rate after preprocessing

print("✅ Plot style configured — dark GitHub-inspired theme")

## 📂 Cell 4 — Data Loading

> **Set `DATA_DIR`** to the folder containing your `s01.dat` … `s32.dat` files.

In [ ]:
# ─── !! CHANGE THIS PATH TO YOUR DEAP FOLDER !! ───────────────────────────────
DATA_DIR = './data'          # e.g. '/kaggle/input/deap-dataset/'
# ──────────────────────────────────────────────────────────────────────────────

def load_deap(data_dir, n_subjects=32):
    """
    Load all DEAP .dat files.
    Returns:
        data   : np.ndarray  shape (n_subjects*40, 40, 8064)  – EEG only (ch 0-31)
        labels : np.ndarray  shape (n_subjects*40, 4)          – valence/arousal/dom/liking
    """
    all_data, all_labels = [], []
    files = sorted(glob.glob(os.path.join(data_dir, 's*.dat')))

    if len(files) == 0:
        raise FileNotFoundError(
            f"No .dat files found in '{data_dir}'.\n"
            "Please set DATA_DIR to the folder containing s01.dat … s32.dat"
        )

    for f in tqdm(files[:n_subjects], desc='Loading subjects'):
        with open(f, 'rb') as fp:
            subject = pickle.load(fp, encoding='latin1')
        all_data.append(subject['data'][:, :32, 3*SRATE:])  # ch 0-31, remove 3s baseline
        all_labels.append(subject['labels'])

    data   = np.vstack(all_data)    # (N_trials, 32, time)
    labels = np.vstack(all_labels)  # (N_trials, 4)
    return data, labels


data, labels = load_deap(DATA_DIR, n_subjects=32)

print("\n" + "═"*55)
print("  DEAP DATASET — LOADED SUCCESSFULLY")
print("═"*55)
print(f"  EEG data shape    : {data.shape}")
print(f"  Labels shape      : {labels.shape}")
print(f"  Sampling rate     : {SRATE} Hz")
print(f"  Duration/trial    : {data.shape[2]/SRATE:.1f} s")
print(f"  Total trials      : {data.shape[0]}")
print(f"  EEG channels      : {data.shape[1]}")
print("─"*55)
print(f"  Labels [0] Valence   – range: [{labels[:,0].min():.1f}, {labels[:,0].max():.1f}]")
print(f"  Labels [1] Arousal   – range: [{labels[:,1].min():.1f}, {labels[:,1].max():.1f}]")
print(f"  Labels [2] Dominance – range: [{labels[:,2].min():.1f}, {labels[:,2].max():.1f}]")
print(f"  Labels [3] Liking    – range: [{labels[:,3].min():.1f}, {labels[:,3].max():.1f}]")
print("═"*55)

## 📊 Cell 5 — EDA: Label Distributions (Valence & Arousal)

In [ ]:
label_names = ['Valence', 'Arousal', 'Dominance', 'Liking']

fig, axes = plt.subplots(2, 4, figsize=(22, 10))
fig.suptitle('EDA — Label Distributions (Continuous Ratings 1–9)', fontsize=16, y=1.01, color='#e6edf3')

for i, (ax, name) in enumerate(zip(axes[0], label_names)):
    vals = labels[:, i]
    ax.hist(vals, bins=30, color=PALETTE[i], edgecolor='#0d1117', alpha=0.85)
    ax.axvline(vals.mean(), color='#ffa657', lw=2, linestyle='--', label=f'Mean: {vals.mean():.2f}')
    ax.axvline(np.median(vals), color='#56d364', lw=2, linestyle=':', label=f'Median: {np.median(vals):.2f}')
    ax.set_title(name, fontsize=13)
    ax.set_xlabel('Rating')
    ax.set_ylabel('Count')
    ax.legend(fontsize=9)

# Valence vs Arousal 2D scatter — the famous circumplex model
ax_scatter = axes[1, 0]
sc = ax_scatter.scatter(labels[:, 0], labels[:, 1],
                         c=labels[:, 0] - labels[:, 1],
                         cmap='coolwarm', alpha=0.6, s=20)
ax_scatter.axvline(5, color='#ffa657', lw=1.5, linestyle='--', alpha=0.6)
ax_scatter.axhline(5, color='#ffa657', lw=1.5, linestyle='--', alpha=0.6)
ax_scatter.set_xlabel('Valence →')
ax_scatter.set_ylabel('Arousal →')
ax_scatter.set_title('Circumplex Emotion Space\n(Valence vs Arousal)', fontsize=12)
# Quadrant labels
for txt, xy in [('HVHA\n(Excited)', (7.5,7.5)),
                ('LVHA\n(Angry)',   (2.5,7.5)),
                ('HVLA\n(Calm)',    (7.5,2.5)),
                ('LVLA\n(Sad)',     (2.5,2.5))]:
    ax_scatter.text(*xy, txt, ha='center', va='center',
                    fontsize=8, color='#8b949e', style='italic')
plt.colorbar(sc, ax=ax_scatter)

# Binary class distribution after median split
threshold = 5.0
val_bin = (labels[:, 0] >= threshold).astype(int)
aro_bin = (labels[:, 1] >= threshold).astype(int)

for ax, y, title, c in zip(
        [axes[1,1], axes[1,2]],
        [val_bin, aro_bin],
        ['Valence Binary (threshold=5)', 'Arousal Binary (threshold=5)'],
        [PALETTE[0], PALETTE[1]]):
    unique, counts = np.unique(y, return_counts=True)
    bars = ax.bar(['Low (<5)', 'High (≥5)'], counts, color=[PALETTE[2], c], edgecolor='#0d1117')
    ax.bar_label(bars, padding=3, fontsize=12, color='#e6edf3')
    ax.set_title(title, fontsize=11)
    ax.set_ylabel('Count')
    for j, (u, cnt) in enumerate(zip(unique, counts)):
        ax.text(j, cnt/2, f'{cnt/sum(counts)*100:.1f}%', ha='center', va='center',
                color='white', fontsize=11, fontweight='bold')

# Label correlation
ax_corr = axes[1, 3]
corr = pd.DataFrame(labels, columns=label_names).corr()
sns.heatmap(corr, ax=ax_corr, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, linecolor='#0d1117',
            cbar_kws={'shrink': 0.8})
ax_corr.set_title('Label Correlation Matrix', fontsize=12)

plt.tight_layout()
plt.savefig('eda_labels.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print("\n📊 Binary class balance:")
print(f"   Valence  — Low: {(val_bin==0).sum()}  High: {(val_bin==1).sum()}")
print(f"   Arousal  — Low: {(aro_bin==0).sum()}  High: {(aro_bin==1).sum()}")

## 📈 Cell 6 — EDA: Raw EEG Signal Visualization

In [ ]:
trial_idx = 0   # change to inspect different trials
n_channels_to_show = 8
time_axis = np.arange(data.shape[2]) / SRATE

fig, axes = plt.subplots(n_channels_to_show, 1, figsize=(20, 14), sharex=True)
fig.suptitle(f'Raw EEG Signal — Trial {trial_idx+1}\n'
             f'Valence={labels[trial_idx,0]:.1f}  Arousal={labels[trial_idx,1]:.1f}',
             fontsize=14, color='#e6edf3')

for i, ax in enumerate(axes):
    ch_data = data[trial_idx, i, :]
    ax.plot(time_axis, ch_data, color=PALETTE[i % len(PALETTE)], lw=0.6, alpha=0.9)
    ax.set_ylabel(CHANNEL_NAMES[i], fontsize=9, rotation=0, labelpad=35, va='center')
    ax.set_ylim(ch_data.mean()-4*ch_data.std(), ch_data.mean()+4*ch_data.std())
    ax.yaxis.set_ticklabels([])

axes[-1].set_xlabel('Time (seconds)', fontsize=11)
plt.tight_layout()
plt.savefig('eda_raw_eeg.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 📉 Cell 7 — EDA: Power Spectral Density (PSD) per Channel

In [ ]:
fig, axes = plt.subplots(4, 8, figsize=(28, 14))
fig.suptitle('Power Spectral Density — All 32 EEG Channels (Trial 0)', fontsize=15)
axes = axes.flatten()

band_colors = {'Delta':'#d2a8ff','Theta':'#79c0ff','Alpha':'#56d364','Beta':'#ffa657','Gamma':'#f78166'}

for i, ax in enumerate(axes):
    freqs, psd = welch(data[trial_idx, i, :], fs=SRATE, nperseg=256)
    ax.semilogy(freqs, psd, color=PALETTE[i%len(PALETTE)], lw=1.2)

    # Shade frequency bands
    for band, (lo, hi) in BANDS.items():
        ax.axvspan(lo, hi, alpha=0.12, color=band_colors[band])

    ax.set_xlim(0, 50)
    ax.set_title(CHANNEL_NAMES[i], fontsize=8, pad=2)
    ax.set_xlabel('', fontsize=7)
    ax.set_ylabel('', fontsize=7)
    ax.tick_params(labelsize=6)

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, alpha=0.4, label=b)
                   for b, c in band_colors.items()]
fig.legend(handles=legend_elements, loc='lower center', ncol=5,
           fontsize=11, bbox_to_anchor=(0.5, -0.02))
plt.tight_layout()
plt.savefig('eda_psd.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 🔥 Cell 8 — EDA: Channel Correlation Heatmap

In [ ]:
# Average correlation across all trials
print("Computing cross-channel correlations across all trials...")
corr_sum = np.zeros((32, 32))
for t in range(min(data.shape[0], 100)):  # use first 100 trials for speed
    corr_sum += np.corrcoef(data[t])
avg_corr = corr_sum / min(data.shape[0], 100)

fig, axes = plt.subplots(1, 2, figsize=(22, 9))
fig.suptitle('EEG Cross-Channel Correlation Analysis', fontsize=15)

# Full heatmap
mask = np.triu(np.ones_like(avg_corr, dtype=bool), k=1)
sns.heatmap(avg_corr, ax=axes[0], cmap='coolwarm', center=0,
            xticklabels=CHANNEL_NAMES, yticklabels=CHANNEL_NAMES,
            linewidths=0.2, linecolor='#0d1117', vmin=-1, vmax=1)
axes[0].set_title('Average Cross-Channel Correlation\n(100 trials)', fontsize=12)
axes[0].tick_params(labelsize=7)

# Distribution of off-diagonal correlations
off_diag = avg_corr[~np.eye(32, dtype=bool)]
axes[1].hist(off_diag, bins=50, color=PALETTE[0], edgecolor='#0d1117', alpha=0.85)
axes[1].axvline(off_diag.mean(), color='#ffa657', lw=2.5, label=f'Mean: {off_diag.mean():.3f}')
axes[1].axvline(0, color='#f78166', lw=1.5, linestyle='--', label='Zero')
axes[1].set_title('Distribution of Off-diagonal Correlations', fontsize=12)
axes[1].set_xlabel('Pearson r')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.savefig('eda_channel_corr.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 🧹 Cell 9 — Preprocessing: Bandpass + Notch Filter

In [ ]:
def bandpass_filter(data_ch, lowcut=4.0, highcut=45.0, fs=128, order=5):
    """4th-order Butterworth bandpass filter (4–45 Hz)"""
    nyq  = 0.5 * fs
    low  = lowcut  / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, data_ch)


def notch_filter(data_ch, notch_freq=50.0, fs=128, quality=30):
    """Notch filter to remove 50 Hz power-line interference"""
    b, a = iirnotch(notch_freq / (0.5 * fs), quality)
    return filtfilt(b, a, data_ch)


def preprocess_eeg(raw_data, fs=128):
    """
    Full preprocessing pipeline:
      1. Bandpass filter  4–45 Hz
      2. Notch filter     50 Hz
      3. Z-score normalize each channel independently
    """
    N, C, T = raw_data.shape
    processed = np.zeros_like(raw_data, dtype=np.float32)
    for n in tqdm(range(N), desc='Preprocessing trials', leave=False):
        for c in range(C):
            ch = raw_data[n, c, :].copy().astype(np.float64)
            ch = bandpass_filter(ch, 4.0, 45.0, fs)
            ch = notch_filter(ch, 50.0, fs)
            # Z-score normalization
            ch = (ch - ch.mean()) / (ch.std() + 1e-8)
            processed[n, c, :] = ch.astype(np.float32)
    return processed


print("⚙️  Preprocessing all trials...")
data_clean = preprocess_eeg(data)
print(f"✅ Preprocessing complete — shape: {data_clean.shape}")

## 🔬 Cell 10 — Preprocessing Visualization: Before vs After

In [ ]:
ch_idx = 0  # Fp1
t = np.arange(data.shape[2]) / SRATE

fig, axes = plt.subplots(3, 2, figsize=(22, 12))
fig.suptitle(f'Preprocessing Effect — Channel: {CHANNEL_NAMES[ch_idx]} (Trial 0)',
             fontsize=14, color='#e6edf3')

raw = data[0, ch_idx, :].astype(np.float64)
clean = data_clean[0, ch_idx, :]

# Time domain
axes[0,0].plot(t, raw, color=PALETTE[2], lw=0.7, alpha=0.9)
axes[0,0].set_title('Raw Signal (Time Domain)')
axes[0,0].set_ylabel('Amplitude (μV)')

axes[0,1].plot(t, clean, color=PALETTE[0], lw=0.7, alpha=0.9)
axes[0,1].set_title('Filtered + Normalised (Time Domain)')
axes[0,1].set_ylabel('Amplitude (z-score)')

# Frequency domain (PSD)
for ax, sig, title, col in [
    (axes[1,0], raw,   'PSD — Raw',     PALETTE[2]),
    (axes[1,1], clean, 'PSD — Filtered', PALETTE[0])
]:
    f, p = welch(sig, fs=SRATE, nperseg=256)
    ax.semilogy(f, p, color=col, lw=1.5)
    for band, (lo, hi) in BANDS.items():
        ax.axvspan(lo, hi, alpha=0.15, color=band_colors[band])
    ax.axvline(50, color='#f78166', lw=1.5, linestyle='--', label='50 Hz notch')
    ax.set_xlim(0, 65)
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('PSD')
    ax.set_title(title)
    ax.legend(fontsize=9)

# Spectrogram
for ax, sig, title, col in [
    (axes[2,0], raw,   'Spectrogram — Raw',     'inferno'),
    (axes[2,1], clean, 'Spectrogram — Filtered', 'viridis')
]:
    f, t_spec, Sxx = signal.spectrogram(sig, fs=SRATE, nperseg=64, noverlap=32)
    ax.pcolormesh(t_spec, f[f<=50], 10*np.log10(Sxx[f<=50]+1e-10),
                  shading='gouraud', cmap=col)
    ax.set_ylabel('Frequency (Hz)')
    ax.set_xlabel('Time (s)')
    ax.set_title(title)
    for _, (lo, hi) in BANDS.items():
        ax.axhline(lo, color='white', lw=0.5, alpha=0.3)

plt.tight_layout()
plt.savefig('preprocessing_comparison.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## ⚡ Cell 11 — Feature Extraction

Extracting a rich feature set covering:
- **Band Power** (5 bands × 32 channels = 160 features)
- **Differential Asymmetry / DASM** (frontal alpha asymmetry, 14 pairs)
- **Statistical Features** per channel (mean, std, skewness, kurtosis, energy = 5 × 32 = 160)
- **Hjorth Parameters** (activity, mobility, complexity = 3 × 32 = 96)

**Total: ~430 handcrafted features per trial**

In [ ]:
def compute_band_power(ch_data, fs=128):
    """Compute average PSD power in each EEG frequency band."""
    freqs, psd = welch(ch_data, fs=fs, nperseg=min(256, len(ch_data)))
    powers = {}
    for band, (lo, hi) in BANDS.items():
        idx = np.logical_and(freqs >= lo, freqs <= hi)
        powers[band] = np.trapz(psd[idx], freqs[idx])
    return powers


def hjorth_params(ch_data):
    """Hjorth activity, mobility, complexity."""
    diff1 = np.diff(ch_data)
    diff2 = np.diff(diff1)
    activity   = np.var(ch_data)
    mobility   = np.sqrt(np.var(diff1) / (np.var(ch_data) + 1e-10))
    complexity = (np.sqrt(np.var(diff2) / (np.var(diff1) + 1e-10))) / (mobility + 1e-10)
    return activity, mobility, complexity


def extract_features(eeg_data, fs=128):
    """
    Extract comprehensive feature vector per trial.
    eeg_data: (n_trials, n_channels, n_timepoints)
    returns : (n_trials, n_features)
    """
    N, C, T = eeg_data.shape
    feature_matrix = []

    for trial_i in tqdm(range(N), desc='Extracting features'):
        feat_vec = []
        band_powers_all = []  # shape (C, 5)

        for ch in range(C):
            sig = eeg_data[trial_i, ch, :]

            # ── Band Powers ───────────────────────────────────────────────────
            bp = compute_band_power(sig, fs)
            band_powers_all.append(list(bp.values()))
            feat_vec.extend(list(bp.values()))

            # ── Statistical Features ──────────────────────────────────────────
            feat_vec.append(np.mean(sig))
            feat_vec.append(np.std(sig))
            feat_vec.append(float(skew(sig)))
            feat_vec.append(float(kurtosis(sig)))
            feat_vec.append(np.sum(sig**2) / T)   # signal energy

            # ── Hjorth Parameters ─────────────────────────────────────────────
            act, mob, comp = hjorth_params(sig)
            feat_vec.extend([act, mob, comp])

        # ── Differential Asymmetry (DASM) — frontal alpha ─────────────────────
        # Pairs: (left, right) in DEAP 10-20 mapping
        # Alpha band index = 2 in BANDS dict
        dasm_pairs = [(0,16),(1,17),(2,19),(3,20),(4,21),(5,22),(6,24),(7,25)]
        bp_arr = np.array(band_powers_all)  # (32, 5)
        alpha_idx = list(BANDS.keys()).index('Alpha')
        for L, R in dasm_pairs:
            feat_vec.append(bp_arr[L, alpha_idx] - bp_arr[R, alpha_idx])  # DASM
            r_right = bp_arr[R, alpha_idx] + 1e-10
            feat_vec.append(bp_arr[L, alpha_idx] / r_right)               # RASM

        feature_matrix.append(feat_vec)

    return np.array(feature_matrix, dtype=np.float32)


print("⚙️  Extracting features from all trials...")
X_features = extract_features(data_clean)
print(f"✅ Feature matrix shape: {X_features.shape}")
print(f"   {X_features.shape[1]} features per trial")

## 🎯 Cell 12 — Label Binarization (Binary Classification)

In [ ]:
# Binary classification: median-split (threshold = 5)
# 0 = Low, 1 = High
THRESHOLD = 5.0
TARGET    = 'valence'   # 'valence' or 'arousal'

label_idx_map = {'valence': 0, 'arousal': 1, 'dominance': 2, 'liking': 3}
idx = label_idx_map[TARGET]

y_val = (labels[:, 0] >= THRESHOLD).astype(int)  # Valence labels
y_aro = (labels[:, 1] >= THRESHOLD).astype(int)  # Arousal labels
y     = y_val  # We'll train for valence; change to y_aro for arousal

print(f"Target: {TARGET.upper()} (binary)")
print(f"Class distribution — 0 (Low): {(y==0).sum()}  |  1 (High): {(y==1).sum()}")
print(f"Class balance     : {(y==0).sum()/len(y)*100:.1f}% / {(y==1).sum()/len(y)*100:.1f}%")

## 📐 Cell 13 — PCA Analysis

In [ ]:
# ── Normalise features before PCA ─────────────────────────────────────────────
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_features)

# Handle NaN / Inf
X_scaled = np.nan_to_num(X_scaled, nan=0.0, posinf=0.0, neginf=0.0)

# ── Full PCA ───────────────────────────────────────────────────────────────────
pca_full = PCA(random_state=SEED)
pca_full.fit(X_scaled)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)

n_95 = np.argmax(cumvar >= 0.95) + 1
n_99 = np.argmax(cumvar >= 0.99) + 1

fig, axes = plt.subplots(1, 3, figsize=(22, 7))
fig.suptitle('Principal Component Analysis (PCA) of EEG Features', fontsize=15)

# Scree plot
axes[0].plot(range(1, min(51, len(pca_full.explained_variance_ratio_)+1)),
             pca_full.explained_variance_ratio_[:50]*100,
             'o-', color=PALETTE[0], lw=2, ms=5)
axes[0].fill_between(range(1, min(51, len(pca_full.explained_variance_ratio_)+1)),
                      pca_full.explained_variance_ratio_[:50]*100,
                      alpha=0.3, color=PALETTE[0])
axes[0].set_title('Scree Plot (Top 50 PCs)', fontsize=12)
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance (%)')

# Cumulative variance
axes[1].plot(range(1, len(cumvar)+1), cumvar*100, color=PALETTE[1], lw=2.5)
axes[1].axhline(95, color='#ffa657', lw=2, linestyle='--', label=f'95% → {n_95} PCs')
axes[1].axhline(99, color='#f78166', lw=2, linestyle='--', label=f'99% → {n_99} PCs')
axes[1].axvline(n_95, color='#ffa657', lw=1.5, linestyle=':')
axes[1].axvline(n_99, color='#f78166', lw=1.5, linestyle=':')
axes[1].set_title('Cumulative Explained Variance', fontsize=12)
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Variance (%)')
axes[1].legend(fontsize=11)

# 2D PCA scatter
pca_2d = PCA(n_components=2, random_state=SEED)
X_pca2 = pca_2d.fit_transform(X_scaled)
for cls, col, lbl in [(0, PALETTE[2], 'Low Valence'), (1, PALETTE[1], 'High Valence')]:
    mask = y == cls
    axes[2].scatter(X_pca2[mask,0], X_pca2[mask,1],
                    c=col, alpha=0.55, s=18, label=lbl, edgecolors='none')
axes[2].set_title('PCA 2D Projection\n(Coloured by Valence Class)', fontsize=12)
axes[2].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}%)')
axes[2].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}%)')
axes[2].legend()

plt.tight_layout()
plt.savefig('pca_analysis.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

print(f"\n📊 PCA Summary:")
print(f"   Original features  : {X_scaled.shape[1]}")
print(f"   PCs for 95% var    : {n_95}")
print(f"   PCs for 99% var    : {n_99}")

# Reduce features using 95% threshold
pca_95 = PCA(n_components=n_95, random_state=SEED)
X_pca  = pca_95.fit_transform(X_scaled)
print(f"   Reduced X_pca shape: {X_pca.shape}")

## 🌐 Cell 14 — t-SNE Visualization

In [ ]:
print("⚙️  Running t-SNE (this may take ~1 min)...")
tsne = TSNE(n_components=2, perplexity=40, n_iter=1000,
            learning_rate='auto', random_state=SEED)
X_tsne = tsne.fit_transform(X_pca[:, :50])  # feed top-50 PCs for speed

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('t-SNE 2D Embedding of EEG Features', fontsize=14)

for ax, label_arr, title in [
    (axes[0], y_val, 'Coloured by VALENCE'),
    (axes[1], y_aro, 'Coloured by AROUSAL')
]:
    for cls, col, lbl in [(0, PALETTE[2], 'Low'), (1, PALETTE[1], 'High')]:
        mask = label_arr == cls
        ax.scatter(X_tsne[mask,0], X_tsne[mask,1],
                   c=col, alpha=0.55, s=18, label=lbl, edgecolors='none')
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('t-SNE Dim 1')
    ax.set_ylabel('t-SNE Dim 2')
    ax.legend(title='Class', fontsize=10)

plt.tight_layout()
plt.savefig('tsne_visualization.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 🌲 Cell 15 — Feature Importance (Random Forest)

In [ ]:
# Build feature names
feat_names = []
for ch in CHANNEL_NAMES:
    for band in BANDS:
        feat_names.append(f'{ch}_{band}')
    for stat in ['Mean','Std','Skew','Kurt','Energy']:
        feat_names.append(f'{ch}_{stat}')
    for hjorth in ['Activity','Mobility','Complexity']:
        feat_names.append(f'{ch}_{hjorth}')
dasm_pairs = [(0,16),(1,17),(2,19),(3,20),(4,21),(5,22),(6,24),(7,25)]
for L, R in dasm_pairs:
    feat_names.append(f'DASM_{CHANNEL_NAMES[L]}_{CHANNEL_NAMES[R]}')
    feat_names.append(f'RASM_{CHANNEL_NAMES[L]}_{CHANNEL_NAMES[R]}')

# Pad to match if sizes differ
while len(feat_names) < X_features.shape[1]:
    feat_names.append(f'feat_{len(feat_names)}')
feat_names = feat_names[:X_features.shape[1]]

print("⚙️  Fitting Random Forest for feature importance...")
rf_imp = RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)
rf_imp.fit(X_scaled, y)
importances = rf_imp.feature_importances_

top_n = 30
top_idx = np.argsort(importances)[-top_n:][::-1]

fig, ax = plt.subplots(figsize=(16, 9))
colors = [PALETTE[i%len(PALETTE)] for i in range(top_n)]
bars = ax.barh(range(top_n), importances[top_idx][::-1], color=colors[::-1], edgecolor='#0d1117')
ax.set_yticks(range(top_n))
ax.set_yticklabels([feat_names[i] for i in top_idx[::-1]], fontsize=9)
ax.set_xlabel('Feature Importance (Gini)', fontsize=12)
ax.set_title(f'Top {top_n} Most Important EEG Features\n(Random Forest, Valence Classification)', fontsize=13)
ax.bar_label(bars, fmt='%.4f', padding=3, fontsize=8, color='#8b949e')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## ✂️ Cell 16 — Train / Validation / Test Split

In [ ]:
# ── Feature-based split (for classical ML) ────────────────────────────────────
X_tr, X_te, y_tr, y_te = train_test_split(
    X_pca, y, test_size=0.20, stratify=y, random_state=SEED)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_tr, y_tr, test_size=0.15, stratify=y_tr, random_state=SEED)

# ── Raw signal split (for deep learning) ──────────────────────────────────────
# Downsample time axis to 256 points for memory efficiency
T_WIN = 256
data_dl = data_clean[:, :, :T_WIN * (data_clean.shape[2]//T_WIN)]
data_dl = data_dl.reshape(data_dl.shape[0], data_dl.shape[1], -1, T_WIN).mean(axis=2)  # avg windows
# Actually let's use a fixed window: last 256×n_windows
# Use the full signal resized to (N, 32, 512) by picking the last 512 pts
data_dl = data_clean[:, :, -512:].transpose(0, 2, 1)  # (N, 512, 32) — time-first for LSTM

X_tr_dl,  X_te_dl,  y_tr_dl,  y_te_dl  = train_test_split(
    data_dl, y, test_size=0.20, stratify=y, random_state=SEED)
X_tr_dl, X_val_dl, y_tr_dl, y_val_dl = train_test_split(
    X_tr_dl, y_tr_dl, test_size=0.15, stratify=y_tr_dl, random_state=SEED)

print("Dataset splits:")
print(f"  Feature-based → Train: {X_tr.shape}  Val: {X_val.shape}  Test: {X_te.shape}")
print(f"  Deep Learning → Train: {X_tr_dl.shape}  Val: {X_val_dl.shape}  Test: {X_te_dl.shape}")

## 🤖 Cell 17 — Classical ML Models: SVM, Random Forest, XGBoost

In [ ]:
results = {}

def train_eval_classical(name, clf, X_tr, y_tr, X_te, y_te):
    clf.fit(X_tr, y_tr)
    y_pred  = clf.predict(X_te)
    y_prob  = clf.predict_proba(X_te)[:,1] if hasattr(clf, 'predict_proba') else None
    acc     = accuracy_score(y_te, y_pred)
    f1      = f1_score(y_te, y_pred, average='weighted')
    auc_sc  = roc_auc_score(y_te, y_prob) if y_prob is not None else None
    print(f"  {name:<20} Acc: {acc:.4f}  F1: {f1:.4f}  AUC: {auc_sc:.4f if auc_sc else 'N/A'}")
    results[name] = {'acc': acc, 'f1': f1, 'auc': auc_sc, 'clf': clf,
                     'y_pred': y_pred, 'y_prob': y_prob}
    return clf


print("\n🔥 Training Classical ML Models...\n")
print(f"{'Model':<20} {'Accuracy':>10}  {'F1-Score':>10}  {'AUC-ROC':>10}")
print("-"*55)

# SVM
svm = SVC(kernel='rbf', C=10, gamma='scale', probability=True, random_state=SEED)
train_eval_classical('SVM (RBF)', svm, X_tr, y_tr, X_te, y_te)

# Random Forest
rf = RandomForestClassifier(n_estimators=300, max_depth=None, random_state=SEED, n_jobs=-1)
train_eval_classical('Random Forest', rf, X_tr, y_tr, X_te, y_te)

# XGBoost
if XGBOOST_AVAILABLE:
    xgb = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                         use_label_encoder=False, eval_metric='logloss',
                         random_state=SEED, n_jobs=-1)
    train_eval_classical('XGBoost', xgb, X_tr, y_tr, X_te, y_te)

## 🧠 Cell 18 — Deep Learning: 1D-CNN

In [ ]:
def build_cnn1d(input_shape, n_classes=2):
    """
    1D-CNN for EEG temporal feature extraction.
    Input: (timesteps, n_channels)
    """
    model = Sequential([
        Input(shape=input_shape),

        # Block 1
        Conv1D(64, kernel_size=7, padding='same', activation='relu'),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),
        Dropout(0.3),

        # Block 2
        Conv1D(128, kernel_size=5, padding='same', activation='relu'),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),
        Dropout(0.3),

        # Block 3
        Conv1D(256, kernel_size=3, padding='same', activation='relu'),
        BatchNormalization(),
        GlobalAveragePooling1D(),
        Dropout(0.4),

        # Dense head
        Dense(128, activation='relu', kernel_regularizer=l2(1e-4)),
        BatchNormalization(),
        Dropout(0.4),
        Dense(64, activation='relu'),
        Dense(1 if n_classes == 2 else n_classes,
              activation='sigmoid' if n_classes == 2 else 'softmax')
    ], name='CNN_1D')
    return model


CALLBACKS = [
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-6, verbose=1)
]

cnn_model = build_cnn1d(input_shape=X_tr_dl.shape[1:])
cnn_model.compile(optimizer=Adam(1e-3),
                   loss='binary_crossentropy',
                   metrics=['accuracy'])
cnn_model.summary()

print("\n🏋️ Training 1D-CNN...")
hist_cnn = cnn_model.fit(
    X_tr_dl, y_tr_dl,
    validation_data=(X_val_dl, y_val_dl),
    epochs=100, batch_size=64,
    callbacks=CALLBACKS, verbose=1
)

## 🔁 Cell 19 — Deep Learning: Bidirectional LSTM

In [ ]:
def build_bilstm(input_shape, n_classes=2):
    """
    Bidirectional LSTM — captures temporal dependencies
    in both directions across the EEG time series.
    """
    model = Sequential([
        Input(shape=input_shape),

        Bidirectional(LSTM(128, return_sequences=True, dropout=0.3, recurrent_dropout=0.2)),
        BatchNormalization(),

        Bidirectional(LSTM(64, return_sequences=False, dropout=0.3, recurrent_dropout=0.2)),
        BatchNormalization(),

        Dense(128, activation='relu', kernel_regularizer=l2(1e-4)),
        Dropout(0.4),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(1, activation='sigmoid')
    ], name='Bi_LSTM')
    return model


bilstm_model = build_bilstm(input_shape=X_tr_dl.shape[1:])
bilstm_model.compile(optimizer=Adam(5e-4),
                      loss='binary_crossentropy',
                      metrics=['accuracy'])
bilstm_model.summary()

print("\n🏋️ Training Bi-LSTM...")
hist_bilstm = bilstm_model.fit(
    X_tr_dl, y_tr_dl,
    validation_data=(X_val_dl, y_val_dl),
    epochs=100, batch_size=64,
    callbacks=CALLBACKS, verbose=1
)

## ⭐ Cell 20 — BEST MODEL: CNN-LSTM Hybrid with Attention

> This is the **state-of-the-art** architecture for EEG emotion recognition.  
> CNN extracts **spatial** features, LSTM captures **temporal** dynamics,  
> and Multi-Head Attention focuses on the most emotionally relevant segments.

In [ ]:
def build_cnn_lstm_attention(input_shape, n_classes=2):
    """
    CNN-LSTM with Multi-Head Self-Attention.
    Architecture:
      [Input] → CNN blocks (spatial/local patterns)
             → Bi-LSTM (temporal dependencies)
             → Multi-Head Attention (focus mechanism)
             → Dense head
             → [Output]
    """
    inputs = Input(shape=input_shape, name='eeg_input')

    # ── CNN Feature Extraction ──────────────────────────────────────────────────
    x = Conv1D(64,  kernel_size=7, padding='same', activation='relu', name='conv1')(inputs)
    x = BatchNormalization()(x)
    x = Dropout(0.2)(x)

    x = Conv1D(128, kernel_size=5, padding='same', activation='relu', name='conv2')(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.2)(x)

    x = Conv1D(256, kernel_size=3, padding='same', activation='relu', name='conv3')(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.25)(x)

    # ── Bi-LSTM Temporal Modelling ──────────────────────────────────────────────
    x = Bidirectional(LSTM(128, return_sequences=True, dropout=0.3), name='bilstm1')(x)
    x = LayerNormalization()(x)

    x = Bidirectional(LSTM(64,  return_sequences=True, dropout=0.25), name='bilstm2')(x)
    x = LayerNormalization()(x)

    # ── Multi-Head Self-Attention ───────────────────────────────────────────────
    attn_out = MultiHeadAttention(num_heads=4, key_dim=32, name='mh_attention')(x, x)
    x = Add()([x, attn_out])                   # Residual connection
    x = LayerNormalization()(x)
    x = GlobalAveragePooling1D()(x)

    # ── Dense Classifier Head ───────────────────────────────────────────────────
    x = Dense(256, activation='relu', kernel_regularizer=l2(1e-4))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.4)(x)
    x = Dense(128, activation='relu', kernel_regularizer=l2(1e-4))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    x = Dense(64, activation='relu')(x)

    outputs = Dense(1, activation='sigmoid', name='output')(x)

    model = Model(inputs, outputs, name='CNN_LSTM_Attention')
    return model


best_model = build_cnn_lstm_attention(input_shape=X_tr_dl.shape[1:])
best_model.compile(
    optimizer=Adam(learning_rate=5e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)
best_model.summary()

callbacks_best = [
    EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=8,
                      min_lr=1e-6, verbose=1),
    ModelCheckpoint('best_eeg_model.h5', monitor='val_accuracy',
                    save_best_only=True, verbose=1)
]

print("\n🏋️ Training CNN-LSTM-Attention (Best Model)...")
hist_best = best_model.fit(
    X_tr_dl, y_tr_dl,
    validation_data=(X_val_dl, y_val_dl),
    epochs=150, batch_size=32,
    callbacks=callbacks_best, verbose=1
)

## 📈 Cell 21 — Training Curves

In [ ]:
def plot_training_history(histories, names):
    fig, axes = plt.subplots(1, 2, figsize=(20, 7))
    fig.suptitle('Deep Learning Model Training Curves', fontsize=15)

    for hist, name, col in zip(histories, names, PALETTE):
        h = hist.history
        eps = range(1, len(h['loss'])+1)

        axes[0].plot(eps, h['loss'],     color=col, lw=2,   label=f'{name} train')
        axes[0].plot(eps, h['val_loss'], color=col, lw=2,
                     linestyle='--', alpha=0.7, label=f'{name} val')

        axes[1].plot(eps, h['accuracy'],     color=col, lw=2,   label=f'{name} train')
        axes[1].plot(eps, h['val_accuracy'], color=col, lw=2,
                     linestyle='--', alpha=0.7, label=f'{name} val')

    for ax, title, ylabel in [
        (axes[0], 'Loss over Epochs',    'Binary Cross-Entropy'),
        (axes[1], 'Accuracy over Epochs','Accuracy')
    ]:
        ax.set_title(title, fontsize=13)
        ax.set_xlabel('Epoch')
        ax.set_ylabel(ylabel)
        ax.legend(fontsize=9, ncol=2)

    plt.tight_layout()
    plt.savefig('training_curves.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
    plt.show()


plot_training_history(
    [hist_cnn, hist_bilstm, hist_best],
    ['1D-CNN', 'Bi-LSTM', 'CNN-LSTM-Attn']
)

## 🏆 Cell 22 — Full Model Evaluation + Confusion Matrices

In [ ]:
def evaluate_dl_model(model, X_te, y_te, name):
    y_prob = model.predict(X_te, verbose=0).flatten()
    y_pred = (y_prob >= 0.5).astype(int)
    acc  = accuracy_score(y_te, y_pred)
    f1   = f1_score(y_te, y_pred, average='weighted')
    auc_ = roc_auc_score(y_te, y_prob)
    print(f"  {name:<25} Acc: {acc:.4f}  F1: {f1:.4f}  AUC: {auc_:.4f}")
    return {'acc': acc, 'f1': f1, 'auc': auc_, 'y_pred': y_pred, 'y_prob': y_prob}

print("\n" + "═"*60)
print("  TEST SET PERFORMANCE")
print("═"*60)
print(f"  {'Model':<25} {'Accuracy':>10}  {'F1':>8}  {'AUC':>8}")
print("-"*60)

cnn_res    = evaluate_dl_model(cnn_model,    X_te_dl, y_te_dl, '1D-CNN')
bilstm_res = evaluate_dl_model(bilstm_model, X_te_dl, y_te_dl, 'Bi-LSTM')
best_res   = evaluate_dl_model(best_model,   X_te_dl, y_te_dl, 'CNN-LSTM-Attention ★')
print("═"*60)

# ── Confusion Matrices ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('Confusion Matrices — Test Set', fontsize=15)

for ax, res, name in [
    (axes[0], cnn_res,    '1D-CNN'),
    (axes[1], bilstm_res, 'Bi-LSTM'),
    (axes[2], best_res,   'CNN-LSTM-Attention ★')
]:
    cm = confusion_matrix(y_te_dl, res['y_pred'])
    cm_pct = cm.astype(float) / cm.sum(axis=1)[:, np.newaxis] * 100
    sns.heatmap(cm_pct, ax=ax, annot=True, fmt='.1f', cmap='Blues',
                linewidths=1, linecolor='#0d1117',
                xticklabels=['Low','High'],
                yticklabels=['Low','High'])
    ax.set_title(f'{name}\nAcc: {res["acc"]:.4f}  AUC: {res["auc"]:.4f}', fontsize=11)
    ax.set_xlabel('Predicted', fontsize=11)
    ax.set_ylabel('True', fontsize=11)

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

# Classification report for best model
print("\n📋 Classification Report — CNN-LSTM-Attention (Best Model):")
print(classification_report(y_te_dl, best_res['y_pred'],
                             target_names=['Low Valence','High Valence']))

## 📉 Cell 23 — ROC-AUC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
ax.plot([0,1],[0,1], 'k--', lw=1.5, alpha=0.5, label='Random Classifier')

all_models = [
    ('SVM (RBF)',           results.get('SVM (RBF)',{}).get('y_prob'), y_te,    PALETTE[0]),
    ('Random Forest',       results.get('Random Forest',{}).get('y_prob'), y_te, PALETTE[1]),
    ('XGBoost',             results.get('XGBoost',{}).get('y_prob'), y_te,       PALETTE[2]),
    ('1D-CNN',              cnn_res['y_prob'],    y_te_dl, PALETTE[3]),
    ('Bi-LSTM',             bilstm_res['y_prob'], y_te_dl, PALETTE[4]),
    ('CNN-LSTM-Attn ★',     best_res['y_prob'],   y_te_dl, PALETTE[5]),
]

for name, y_prob, y_true, col in all_models:
    if y_prob is None:
        continue
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_val = auc(fpr, tpr)
    lw = 3.5 if '★' in name else 2
    ax.plot(fpr, tpr, color=col, lw=lw, label=f'{name}  (AUC = {auc_val:.4f})')

ax.set_xlabel('False Positive Rate', fontsize=13)
ax.set_ylabel('True Positive Rate',  fontsize=13)
ax.set_title('ROC-AUC Curves — All Models (Valence Classification)', fontsize=14)
ax.legend(fontsize=10, loc='lower right')
ax.set_xlim([-0.01, 1.01])
ax.set_ylim([-0.01, 1.05])
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 🏅 Cell 24 — Final Model Comparison Table

In [ ]:
summary_data = []

# Classical models
for name in ['SVM (RBF)', 'Random Forest', 'XGBoost']:
    r = results.get(name)
    if r:
        summary_data.append({'Model': name, 'Type': 'Classical ML',
                              'Accuracy': f"{r['acc']*100:.2f}%",
                              'F1-Score': f"{r['f1']:.4f}",
                              'AUC-ROC':  f"{r['auc']:.4f}" if r['auc'] else 'N/A'})

# Deep Learning
for name, res in [('1D-CNN', cnn_res), ('Bi-LSTM', bilstm_res),
                   ('CNN-LSTM-Attention ★', best_res)]:
    summary_data.append({'Model': name, 'Type': 'Deep Learning',
                          'Accuracy': f"{res['acc']*100:.2f}%",
                          'F1-Score': f"{res['f1']:.4f}",
                          'AUC-ROC':  f"{res['auc']:.4f}"})

df_summary = pd.DataFrame(summary_data)

# Styled table
fig, ax = plt.subplots(figsize=(14, len(df_summary)*0.7 + 1.5))
ax.axis('off')
table = ax.table(
    cellText=df_summary.values,
    colLabels=df_summary.columns,
    cellLoc='center', loc='center'
)
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.4, 2.2)

# Header style
for j in range(len(df_summary.columns)):
    table[0, j].set_facecolor('#1f6feb')
    table[0, j].set_text_props(color='white', fontweight='bold')

# Row style
for i in range(1, len(df_summary)+1):
    row_color = '#21262d' if i % 2 == 0 else '#161b22'
    if '★' in df_summary.iloc[i-1]['Model']:
        row_color = '#0d4429'  # highlight best model
    for j in range(len(df_summary.columns)):
        table[i, j].set_facecolor(row_color)
        table[i, j].set_text_props(color='#e6edf3')

ax.set_title('🏆 Model Performance Comparison — EEG Emotion Recognition (Valence)',
             fontsize=13, color='#e6edf3', pad=20)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

print(df_summary.to_string(index=False))

## 🔬 Cell 25 — Band Power Analysis: Emotion-wise Mean Power Topography

In [ ]:
# Mean band power per class per channel
band_list = list(BANDS.keys())

# Reshape features to extract band powers (first 160 features = 32ch × 5 bands)
bp_feats = X_features[:, :160].reshape(-1, 32, 5)  # (N, 32, 5)

fig, axes = plt.subplots(2, 5, figsize=(24, 10))
fig.suptitle('Mean Band Power per Channel — Low vs High Valence', fontsize=15)

for b_idx, band_name in enumerate(band_list):
    for class_idx, (cls_name, ax) in enumerate([
        ('Low Valence',  axes[0, b_idx]),
        ('High Valence', axes[1, b_idx])
    ]):
        mask = y_val == class_idx
        mean_power = bp_feats[mask, :, b_idx].mean(axis=0)
        norm_power = (mean_power - mean_power.min()) / (mean_power.ptp() + 1e-10)

        bars = ax.bar(range(32), norm_power,
                      color=plt.cm.coolwarm(norm_power), edgecolor='none')
        ax.set_xticks(range(32))
        ax.set_xticklabels(CHANNEL_NAMES, rotation=90, fontsize=6)
        ax.set_title(f'{band_name} — {cls_name}', fontsize=9)
        ax.set_ylabel('Norm. Power', fontsize=8)

plt.tight_layout()
plt.savefig('band_power_topology.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## ✅ Cell 26 — Summary & Conclusions

In [ ]:
best_acc = best_res['acc']
best_auc = best_res['auc']
best_f1  = best_res['f1']

print("""
╔══════════════════════════════════════════════════════════════════╗
║       EEG EMOTION RECOGNITION — PIPELINE SUMMARY                ║
╠══════════════════════════════════════════════════════════════════╣
║  Dataset      : DEAP (32 subjects × 40 trials × 32 EEG ch)     ║
║  Target       : Valence (binary: Low / High at threshold=5)     ║
╠══════════════════════════════════════════════════════════════════╣
║  PIPELINE STEPS COMPLETED:                                       ║
║  ✅ Data loading & structure exploration                         ║
║  ✅ EDA — label distributions, circumplex model                 ║
║  ✅ EDA — raw EEG visualization                                 ║
║  ✅ EDA — PSD per channel (welch method)                        ║
║  ✅ EDA — cross-channel correlation heatmap                     ║
║  ✅ Preprocessing — bandpass filter (4–45 Hz)                   ║
║  ✅ Preprocessing — notch filter (50 Hz)                        ║
║  ✅ Preprocessing — z-score normalisation                       ║
║  ✅ Feature extraction — 5 band powers (δθαβγ)                  ║
║  ✅ Feature extraction — statistical moments                    ║
║  ✅ Feature extraction — Hjorth parameters                      ║
║  ✅ Feature extraction — DASM / RASM asymmetry                  ║
║  ✅ PCA — scree, cumulative variance, 2D scatter                ║
║  ✅ t-SNE — 2D embedding                                        ║
║  ✅ Feature importance — Random Forest                          ║
║  ✅ Classical ML — SVM, Random Forest, XGBoost                  ║
║  ✅ Deep Learning — 1D-CNN                                      ║
║  ✅ Deep Learning — Bidirectional LSTM                          ║
║  ✅ Deep Learning — CNN-LSTM + Multi-Head Attention (BEST)      ║
║  ✅ Confusion matrices, ROC-AUC, classification reports         ║
║  ✅ Band power topology analysis                                ║
╠══════════════════════════════════════════════════════════════════╣
║  BEST MODEL: CNN-LSTM with Multi-Head Attention                 ║"""
f"""
║  ► Test Accuracy : {best_acc*100:.2f}%                                       ║
║  ► AUC-ROC       : {best_auc:.4f}                                     ║
║  ► F1-Score      : {best_f1:.4f}                                     ║
╠══════════════════════════════════════════════════════════════════╣
║  KEY FINDINGS (grounded in survey paper):                        ║
║  • Gamma & Beta bands most discriminative for emotion           ║
║  • Frontal alpha asymmetry (DASM/RASM) is emotion-predictive    ║
║  • CNN-LSTM outperforms classical ML as per survey literature   ║
║  • Bi-LSTM captures long-range temporal EEG dependencies        ║
║  • Attention mechanism aligns with emotion-relevant EEG segs.   ║
╚══════════════════════════════════════════════════════════════════╝
""")